# Public Sentiment Analysis Pipeline

GitHub-safe Colab notebook using synthetic data and the same RoBERTa sentiment-scoring methodology.

In [ ]:
# Install dependencies
!pip -q install pandas openpyxl torch transformers tqdm scikit-learn

In [ ]:
# Optional: mount Google Drive if you want to save outputs there
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
from pathlib import Path
import pandas as pd
from tqdm.notebook import tqdm

# If running from the project root in Colab, this path should work.
DATA_PATH = Path('/content/public_sentiment_roberta_project/data/synthetic_customer_feedback.csv')

# Alternative: upload the CSV manually and change this path:
# DATA_PATH = Path('/content/synthetic_customer_feedback.csv')

df = pd.read_csv(DATA_PATH)
df.head()

In [ ]:
import re

TEXT_COLUMN = 'translated text'

def remove_bracketed_text(text):
    return re.sub(r'\[[^\[\]]*\]', '', str(text)).strip()

def clean_translated_text(text):
    text = str(text)
    text = remove_bracketed_text(text)
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = re.sub(r'\S+@\S+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df = df[df[TEXT_COLUMN].notna()]
df = df[df[TEXT_COLUMN].apply(lambda x: isinstance(x, str))].reset_index(drop=True)
df = df[~df[TEXT_COLUMN].isin(['none', 'None', 'Blank', 'blank', 'no', ''])]
df['Translated_text_cleaned'] = df[TEXT_COLUMN].apply(clean_translated_text)
df = df[df['Translated_text_cleaned'].str.len() > 0].reset_index(drop=True)

df[[TEXT_COLUMN, 'Translated_text_cleaned']].head()

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

MODEL_NAME = "cardiffnlp/twitter-roberta-base-sentiment"
LABELS = ["Negative", "Neutral", "Positive"]

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME)
model.eval()

def predict_sentiment(text):
    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True,
        max_length=512
    )
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.nn.functional.softmax(outputs.logits, dim=1).squeeze().tolist()
    pred_idx = int(torch.argmax(torch.tensor(probs)).item())
    return {
        "Predicted_Sentiment": LABELS[pred_idx],
        "Negative_Probability": round(float(probs[0]), 6),
        "Neutral_Probability": round(float(probs[1]), 6),
        "Positive_Probability": round(float(probs[2]), 6),
        "Confidence": round(float(max(probs)), 6),
    }

In [ ]:
tqdm.pandas()

predictions = df['Translated_text_cleaned'].progress_apply(predict_sentiment)
pred_df = pd.DataFrame(list(predictions))
output_df = pd.concat([df, pred_df], axis=1)

output_df.head()

In [ ]:
# Simple quality check for the synthetic demo label only
if 'Expected_Sentiment_For_Demo' in output_df.columns:
    from sklearn.metrics import classification_report, confusion_matrix
    print(classification_report(output_df['Expected_Sentiment_For_Demo'], output_df['Predicted_Sentiment']))
    display(pd.DataFrame(
        confusion_matrix(output_df['Expected_Sentiment_For_Demo'], output_df['Predicted_Sentiment'], labels=LABELS),
        index=[f'Actual_{x}' for x in LABELS],
        columns=[f'Pred_{x}' for x in LABELS]
    ))

In [ ]:
# Save output
OUTPUT_DIR = Path('/content/outputs')
OUTPUT_DIR.mkdir(exist_ok=True)

excel_path = OUTPUT_DIR / 'sentiment_predictions.xlsx'
csv_path = OUTPUT_DIR / 'sentiment_predictions.csv'

output_df.to_excel(excel_path, index=False)
output_df.to_csv(csv_path, index=False)

print(f'Saved Excel: {excel_path}')
print(f'Saved CSV: {csv_path}')

In [ ]:
# Optional: save to Google Drive
# drive_output = Path('/content/drive/MyDrive/sentiment_analysis_public_output')
# drive_output.mkdir(parents=True, exist_ok=True)
# output_df.to_excel(drive_output / 'sentiment_predictions.xlsx', index=False)
# output_df.to_csv(drive_output / 'sentiment_predictions.csv', index=False)